# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal check 1 — CTR vs. position tier (flag-linked: behind the CTR-fix logic)

**Verdict: CONFIRMED.** Bucketed by `position_tier`, mean CTR drops monotonically as position
gets worse — 2.76% (top_3) → 0.65% (page_1) → 0.32% (striking) → 0.22% (page_3_5) → 0.15%
(deep), n = 1,116 / 11,814 / 7,304 / 7,242 / 1,319. This is the direct justification for
comparing a page's CTR against its own tier's typical CTR rather than a flat number — exactly
the logic behind the CTR-fix flag from the session.

### Signal check 2 — Staleness vs. relative CTR performance (flag-linked: behind the refresh flags)

**Verdict: MIXED — and this negative saved my rule from a weak signal.** Bucketed by
`freshness_tier`, mean `ctr_gap` (positive = underperforming its tier) does *not* move cleanly
with staleness: 0-30 days = -0.45 (n=19,300), 31-90 days = +0.05 (n=175), 91-180 days = -0.07
(n=9,162), 181+ days = -3.66 (n=158). The two large buckets (0-30, 91-180, together 95% of
eligible rows) show only a weak, near-zero relationship, and the two tiny buckets flip sign
entirely — a sign that a handful of outliers are driving the extreme means, not a real pattern.
Medians tell a slightly different story (181+ has the highest median gap, 0.17, consistent with
staleness mattering), so mean and median disagree — the honest read is: staleness is *not* a
reliable second signal on this slice, at least not on its own. I am deliberately **not** folding
it into the rule below.

### The rule, in plain words

"A page is worth a title/meta rewrite if it already gets real search traffic (≥500 impressions
over 90 days), sits within realistic snippet-editing range (avg position ≤20), and its CTR sits
below what other pages at its own position tier typically achieve." Score = the size of that
CTR gap × how much traffic the page already gets — so a page with a small gap but huge volume
can outrank a page with a big gap but tiny volume, on purpose.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/UnsoundMouse/flyrankaiw01_research_question/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)].copy()

# SIGNAL 1: CTR vs position tier
print("=== SIGNAL 1: CTR vs position tier (flag-linked: CTR-fix logic) ===")
sig1 = eligible.groupby("position_tier").agg(
    n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median")
).round(3)
print(sig1)
print("VERDICT: CONFIRMED -- mean CTR drops monotonically as position worsens.\n")

# tier-median CTR (used for the rule below) computed on visible pages only
visible = eligible[eligible["impressions_90d"] >= 500]
tier_median = visible.groupby("position_tier")["ctr"].median()
eligible = eligible.merge(tier_median.rename("tier_median_ctr"), left_on="position_tier", right_index=True)
eligible["ctr_gap"] = eligible["tier_median_ctr"] - eligible["ctr"]

# SIGNAL 2: staleness vs relative CTR performance
print("=== SIGNAL 2: freshness_tier vs ctr_gap (flag-linked: refresh flags) ===")
sig2 = eligible.groupby("freshness_tier").agg(
    n=("ctr_gap", "size"), mean_gap=("ctr_gap", "mean"), median_gap=("ctr_gap", "median")
).round(4)
print(sig2)
print("VERDICT: MIXED -- mean and median disagree, large buckets show a near-zero effect.")
print("Deliberately excluded from the rule below.")

=== SIGNAL 1: CTR vs position tier (flag-linked: CTR-fix logic) ===
                   n  mean_ctr  median_ctr
position_tier                             
deep            1319     0.150        0.00
page_1         11814     0.652        0.16
page_3_5        7242     0.222        0.03
striking        7304     0.323        0.11
top_3           1116     2.764        0.00
VERDICT: CONFIRMED -- mean CTR drops monotonically as position worsens.

=== SIGNAL 2: freshness_tier vs ctr_gap (flag-linked: refresh flags) ===
                    n  mean_gap  median_gap
freshness_tier                             
0-30            19300   -0.4545        0.07
181+              158   -3.6625        0.17
31-90             175    0.0498        0.09
91-180           9162   -0.0731        0.04
VERDICT: MIXED -- mean and median disagree, large buckets show a near-zero effect.
Deliberately excluded from the rule below.


## 2. Build the ranked queue (writes the CSV)

Score = `ctr_gap × impressions_90d`, gated to `impressions_90d >= 500`, `avg_position <= 20`,
`ctr_gap > 0`. Reason code: `tier_ctr_underperform`. Action: `rewrite_title_meta`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

gated = eligible[
    (eligible["impressions_90d"] >= 500)
    & (eligible["avg_position"] <= 20)
    & (eligible["ctr_gap"] > 0)
].copy()

gated["score"] = gated["ctr_gap"] * gated["impressions_90d"]
gated["reason_code"] = "tier_ctr_underperform"
gated["action"] = "rewrite_title_meta"
gated = gated.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["client_id", "content_id", "content_type", "main_intent", "position_tier",
            "avg_position", "impressions_90d", "clicks_90d", "ctr", "tier_median_ctr",
            "ctr_gap", "score", "reason_code", "action"]
gated[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue written: {len(gated):,} rows -> work/outputs/baseline_action_score.csv")
print(f"Clients represented: {gated['client_id'].nunique()} of {df['client_id'].nunique()}")

Ranked queue written: 5,885 rows -> work/outputs/baseline_action_score.csv
Clients represented: 24 of 32


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(gated[out_cols[:11]].head(20).to_string(index=False))

        client_id           content_id    content_type   main_intent position_tier  avg_position  impressions_90d  clicks_90d  ctr  tier_median_ctr  ctr_gap
client_19581e27de content_36ff89c8214e keyword article informational        page_1           7.3           295097         154 0.05             0.24     0.19
client_4e07408562 content_5fe46e04994d keyword article informational        page_1           4.2           517715         741 0.14             0.24     0.10
client_19581e27de content_c8e9d6ab9013 keyword article informational        page_1           9.7           208678           0 0.00             0.24     0.24
client_f369cb89fc content_c84a0ab98e90 keyword article informational        page_1           7.8           223271          70 0.03             0.24     0.21
client_d029fa3a95 content_8451fc6f034d keyword article informational         top_3           2.3           272144          75 0.03             0.20     0.17
client_f369cb89fc content_453722754fea keyword article inf

Reading the top 20 by hand (all `reason_code = tier_ctr_underperform`, `action = rewrite_title_meta`):

1. **295,097 impr, page_1 pos 7.3, ctr 0.05 vs tier 0.24** — big, high-confidence gap on real
   traffic. Wrong if: the title already accurately reflects strong intent-match and the gap is
   really a slow/broken page issue, not a metadata issue.
2. **517,715 impr, page_1 pos 4.2, ctr 0.14 vs tier 0.24** — huge volume, moderate gap.
   Wrong if: this SERP shows a rich result (featured snippet, PAA) suppressing clicks regardless
   of title quality — no rewrite fixes that.
3. **208,678 impr, page_1 pos 9.7, ctr 0.00 (zero clicks)** — a real red flag. Wrong if: this is
   a brand-new page still accumulating its first clicks and 90 days isn't representative yet.
4. **223,271 impr, page_1 pos 7.8, ctr 0.03 vs 0.24** — large gap, solid volume. Wrong if: query
   intent is informational-only and users are satisfied by the snippet without clicking through.
5. **272,144 impr, top_3 pos 2.3, ctr 0.03 vs 0.20** — flagged even at top_3, the best tier.
   Wrong if: top_3 pages naturally get some "no-click" branded searches that shouldn't count
   against this page specifically.
6. **140,079 impr, page_1 pos 7.6, ctr 0.01 vs 0.24** — near-total gap. Wrong if: `clicks_90d`
   undercounts due to tracking/consent-mode gaps rather than genuine low interest.
7. **213,963 impr, page_1 pos 4.7, ctr 0.10 vs 0.24** — good position, still underperforming.
   Wrong if: the page ranks for a broad/ambiguous query and much of the impression volume isn't
   actually relevant to the page's real intent.
8. **159,590 impr, page_1 pos 7.8, ctr 0.06 vs 0.24, commercial intent** — commercial intent
   pages often have naturally lower CTR (comparison shopping); tier median may not fully account
   for intent. Wrong if: commercial-intent pages just have a structurally lower ceiling than the
   page_1 median used here.
9. **134,055 impr, page_1 pos 7.5, ctr 0.03 vs 0.24, commercial intent** — same intent caveat as
   #8. Wrong if: intent-adjusted expectations would put this page much closer to normal.
10. **119,217 impr, page_1 pos 7.0, ctr 0.02 vs 0.24** — large gap, real volume. Wrong if: this
    page recently had a title change already in progress and the CTR hasn't caught up in the
    90-day window yet.
11. **201,111 impr, page_1 pos 5.7, ctr 0.11 vs 0.24** — solid position, moderate gap. Wrong if:
    this query has heavy AI-overview coverage now, permanently suppressing CTR regardless of
    title quality.
12. **123,469 impr, page_1 pos 8.0, ctr 0.03 vs 0.24, transactional intent** — same intent caveat
    as #8/#9, now on a transactional query. Wrong if: transactional queries convert via a
    different SERP feature (shopping carousel, ads) that a title rewrite can't touch.
13. **112,434 impr, page_1 pos 7.2, ctr 0.01 vs 0.24** — near-zero CTR, real volume. Wrong if:
    the URL itself looks untrustworthy or spammy in the SERP regardless of title text.
14. **509,252 impr, top_3 pos 2.5, ctr 0.15 vs 0.20** — smallest gap in the top 20 but largest
    volume; scored high mainly because of impressions. Wrong if: a 0.05-point gap at top_3 is
    within normal day-to-day noise and not worth editorial time at all.
15. **147,670 impr, page_1 pos 6.4, ctr 0.07 vs 0.24, transactional intent** — same caveat as
    #12. Wrong if: intent-adjusted comparison would shrink this gap substantially.
16. **309,910 impr, page_1 pos 5.6, ctr 0.16 vs 0.24, transactional intent** — good position,
    moderate gap, high volume. Wrong if: this page already converts well despite modest CTR, so
    "opportunity" here is really just noise, not lost revenue.
17. **128,068 impr, top_3 pos 2.2, ctr 0.01 vs 0.20** — very low CTR at an excellent position,
    stands out. Wrong if: this is a branded/navigational query where users recognize the result
    without needing to click (e.g. it's already the expected top result).
18. **127,952 impr, page_1 pos 7.4, ctr 0.07 vs 0.24, transactional intent** — same caveat as
    #12/#15. Wrong if: intent adjustment would move this well within normal range.
19. **99,013 impr, page_1 pos 6.4, ctr 0.03 vs 0.24** — solid gap, lower volume than most of the
    list. Wrong if: this is right at the edge of the 500-impression filter and one slow week of
    traffic could push it below the eligibility threshold entirely.
20. **149,712 impr, top_3 pos 2.9, ctr 0.07 vs 0.20** — moderate gap at a strong position. Wrong
    if: top_3 pages structurally see more "position-only" impressions (users scanning without
    reading) that a title change won't convert to clicks.

**Pattern across the top 20:** almost every entry is `content_type = keyword article`, and the
list splits fairly evenly across `informational`, `commercial`, and `transactional` intent once
you get past the top 10 — which sharpens the earlier caveat about #8/#9. Six of the twenty
(#8, #9, #12, #15, #16, #18) carry the same weak spot: the rule compares each page's CTR against
a `position_tier` median that's dominated by informational-intent pages, so commercial and
transactional pages may be getting flagged for having a structurally different (not necessarily
"broken") CTR ceiling. That's a real, repeatable weakness worth fixing before Week 5, not a
one-off.

## 4. Weak picks + leakage check

**Weakest picks: #8, #9, #12, #15, #16, #18** — six of twenty, all commercial or transactional
intent, all flagged using a `position_tier` median that's dominated by informational-intent
pages. If commercial/transactional pages structurally get fewer clicks at the same position,
comparing them against a mostly-informational tier median systematically over-flags them. A fix
worth testing next: compute `tier_median_ctr` within `(position_tier, main_intent)` pairs
instead of `position_tier` alone.

**Leakage check:** the rule uses only `impressions_90d`, `avg_position`, `position_tier`, `ctr`,
and `clicks_90d` — all observed at the same snapshot, none derived from a future window or from
the label I'd eventually validate against. No `trend_pct`/`trend_direction` (label-trap columns)
were used anywhere in scoring. No future-window inputs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Two signal verdicts with visible bucket tables and n (both flag-linked: CTR-vs-position CONFIRMED, staleness MIXED)
- [x] One rule with a score, a reason code (`tier_ctr_underperform`), and an action label (`rewrite_title_meta`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived inputs
- [x] No client names, URLs, or private queries anywhere
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.